<a href="https://colab.research.google.com/github/charith7287/pdf-compliance-extractor-power-sector-compliance-tool/blob/main/project_marked_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install streamlit pyngrok pdfplumber pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.7 MB/s eta 0:00:00


pytesseract


In [ ]:
!pip install pytesseract


In [ ]:
!apt-get install -y tesseract-ocr


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
import streamlit as st
import pdfplumber
import pandas as pd
import re
import matplotlib.pyplot as plt
import pytesseract
from pdf2image import convert_from_bytes

st.set_page_config(page_title="AI Compliance Dashboard", layout="wide")
st.title("✅ AI-Enabled Compliance Automation Dashboard")

# ---------------- PDF EXTRACTION ----------------
def extract_text_pdfplumber(uploaded_file):
    text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

# ---------------- OCR EXTRACTION ----------------
def extract_text_ocr(file_bytes):
    images = convert_from_bytes(file_bytes)
    text = ""
    for img in images:
        text += pytesseract.image_to_string(img) + "\n"
    return text.strip()

# ---------------- RISK LEVEL ----------------
def detect_risk(due_date):
    return "High" if due_date != "Not Found" else "Medium"

# ---------------- NLP / COMPLIANCE EXTRACTION ----------------
def extract_compliance_info(text):
    lower_text = text.lower()

    if "report" in lower_text:
        compliance_type = "Reporting"
    elif "audit" in lower_text:
        compliance_type = "Audit"
    elif "safety" in lower_text:
        compliance_type = "Safety"
    else:
        compliance_type = "General"

    if "distribution" in lower_text:
        entity = "Distribution Company"
    elif "transmission" in lower_text:
        entity = "Transmission Utility"
    else:
        entity = "Power Stakeholder"

    date_pattern = r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})"
    match = re.search(date_pattern, text)
    due_date = match.group(1) if match else "Not Found"

    return {
        "compliance_type": compliance_type,
        "entity": entity,
        "due_date": due_date,
        "risk_level": detect_risk(due_date),
        "summary": text[:250] + "..."
    }

# ---------------- SIDEBAR ----------------
st.sidebar.header("⚙️ Settings")
enable_ocr = st.sidebar.checkbox("Enable OCR for scanned PDFs", value=True)

# ---------------- FILE UPLOAD ----------------
uploaded_files = st.file_uploader(
    "📄 Upload Regulatory PDF Documents",
    type=["pdf"],
    accept_multiple_files=True
)

if uploaded_files:
    results = []

    for file in uploaded_files:
        st.write("📌 Processing:", file.name)

        extracted_text = extract_text_pdfplumber(file)

        if enable_ocr and len(extracted_text) < 50:
            extracted_text = extract_text_ocr(file.getvalue())

        if len(extracted_text) < 30:
            st.error("❌ Unable to extract text")
            continue

        compliance_data = extract_compliance_info(extracted_text)

        results.append({
            "document_name": file.name,
            **compliance_data
        })

    # ================== MAIN OUTPUT ==================
    if results:
        df = pd.DataFrame(results)

        # ---- TABLE ----
        st.subheader("📊 Compliance Insights Table")
        st.dataframe(df, use_container_width=True)

        # ---- FILTERS ----
        st.subheader("🔍 Filters")
        col1, col2 = st.columns(2)

        with col1:
            filter_type = st.selectbox(
                "Select Compliance Type",
                ["All"] + sorted(df["compliance_type"].unique())
            )

        with col2:
            filter_risk = st.selectbox(
                "Select Risk Level",
                ["All"] + sorted(df["risk_level"].unique())
            )

        filtered_df = df.copy()

        if filter_type != "All":
            filtered_df = filtered_df[filtered_df["compliance_type"] == filter_type]

        if filter_risk != "All":
            filtered_df = filtered_df[filtered_df["risk_level"] == filter_risk]

        st.subheader("✅ Filtered Results")
        st.dataframe(filtered_df, use_container_width=True)

        # =================================================
        # ✅ INSERTED CHART CODE (CORRECT LOCATION)
        # =================================================
        st.subheader("📈 Compliance Analytics Charts")

        colA, colB = st.columns(2)

        # Chart 1: Compliance Type
        with colA:
            st.write("### Compliance Type Distribution")
            type_counts = df["compliance_type"].value_counts()
            fig1 = plt.figure()
            plt.bar(type_counts.index, type_counts.values)
            plt.ylabel("Count")
            plt.xticks(rotation=20)
            st.pyplot(fig1)

        # Chart 2: Risk Level
        with colB:
            st.write("### Risk Level Distribution")
            risk_counts = df["risk_level"].value_counts()
            fig2 = plt.figure()
            plt.bar(risk_counts.index, risk_counts.values)
            plt.ylabel("Count")
            plt.xticks(rotation=20)
            st.pyplot(fig2)

        # ---- CSV DOWNLOAD ----
        st.subheader("⬇️ Download Output")
        csv = filtered_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            "Download Compliance Report CSV",
            csv,
            file_name="compliance_report.csv",
            mime="text/csv"
        )
else:
    st.info("📌 Upload PDF files to begin compliance extraction.")


2026-01-27 17:13:49.956 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.962 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.963 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.964 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-27 17:13:49.967 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
!pkill -f streamlit
!pkill -f ngrok

In [ ]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
%%writefile app.py
import streamlit as st
import pdfplumber
import pandas as pd
import re
import matplotlib.pyplot as plt
import pytesseract
from pdf2image import convert_from_bytes

st.set_page_config(page_title="AI Compliance Dashboard", layout="wide")
st.title("✅ AI-Enabled Compliance Automation Dashboard")

# ---------------- PDF EXTRACTION ----------------
def extract_text_pdfplumber(uploaded_file):
    text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

# ---------------- OCR EXTRACTION ----------------
def extract_text_ocr(file_bytes):
    images = convert_from_bytes(file_bytes)
    text = ""
    for img in images:
        text += pytesseract.image_to_string(img) + "\n"
    return text.strip()

# ---------------- RISK LEVEL ----------------
def detect_risk(due_date):
    return "High" if due_date != "Not Found" else "Medium"

# ---------------- NLP / COMPLIANCE EXTRACTION ----------------
def extract_compliance_info(text):
    lower_text = text.lower()

    if "report" in lower_text:
        compliance_type = "Reporting"
    elif "audit" in lower_text:
        compliance_type = "Audit"
    elif "safety" in lower_text:
        compliance_type = "Safety"
    else:
        compliance_type = "General"

    if "distribution" in lower_text:
        entity = "Distribution Company"
    elif "transmission" in lower_text:
        entity = "Transmission Utility"
    else:
        entity = "Power Stakeholder"

    date_pattern = r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})"
    match = re.search(date_pattern, text)
    due_date = match.group(1) if match else "Not Found"

    return {
        "compliance_type": compliance_type,
        "entity": entity,
        "due_date": due_date,
        "risk_level": detect_risk(due_date),
        "summary": text[:250] + "..."
    }

# ---------------- SIDEBAR ----------------
st.sidebar.header("⚙️ Settings")
enable_ocr = st.sidebar.checkbox("Enable OCR for scanned PDFs", value=True)

# ---------------- FILE UPLOAD ----------------
uploaded_files = st.file_uploader(
    "📄 Upload Regulatory PDF Documents",
    type=["pdf"],
    accept_multiple_files=True
)

if uploaded_files:
    results = []

    for file in uploaded_files:
        st.write("📌 Processing:", file.name)

        extracted_text = extract_text_pdfplumber(file)

        if enable_ocr and len(extracted_text) < 50:
            extracted_text = extract_text_ocr(file.getvalue())

        if len(extracted_text) < 30:
            st.error("❌ Unable to extract text")
            continue

        compliance_data = extract_compliance_info(extracted_text)

        results.append({
            "document_name": file.name,
            **compliance_data
        })

    # ================== MAIN OUTPUT ==================
    if results:
        df = pd.DataFrame(results)

        # ---- TABLE ----
        st.subheader("📊 Compliance Insights Table")
        st.dataframe(df, use_container_width=True)

        # ---- FILTERS ----
        st.subheader("🔍 Filters")
        col1, col2 = st.columns(2)

        with col1:
            filter_type = st.selectbox(
                "Select Compliance Type",
                ["All"] + sorted(df["compliance_type"].unique())
            )

        with col2:
            filter_risk = st.selectbox(
                "Select Risk Level",
                ["All"] + sorted(df["risk_level"].unique())
            )

        filtered_df = df.copy()

        if filter_type != "All":
            filtered_df = filtered_df[filtered_df["compliance_type"] == filter_type]

        if filter_risk != "All":
            filtered_df = filtered_df[filtered_df["risk_level"] == filter_risk]

        st.subheader("✅ Filtered Results")
        st.dataframe(filtered_df, use_container_width=True)

        # =================================================
        # ✅ INSERTED CHART CODE (CORRECT LOCATION)
        # =================================================
        st.subheader("📈 Compliance Analytics Charts")

        colA, colB = st.columns(2)

        # Chart 1: Compliance Type
        with colA:
            st.write("### Compliance Type Distribution")
            type_counts = df["compliance_type"].value_counts()
            fig1 = plt.figure()
            plt.bar(type_counts.index, type_counts.values)
            plt.ylabel("Count")
            plt.xticks(rotation=20)
            st.pyplot(fig1)

        # Chart 2: Risk Level
        with colB:
            st.write("### Risk Level Distribution")
            risk_counts = df["risk_level"].value_counts()
            fig2 = plt.figure()
            plt.bar(risk_counts.index, risk_counts.values)
            plt.ylabel("Count")
            plt.xticks(rotation=20)
            st.pyplot(fig2)

        # ---- CSV DOWNLOAD ----
        st.subheader("⬇️ Download Output")
        csv = filtered_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            "Download Compliance Report CSV",
            csv,
            file_name="compliance_report.csv",
            mime="text/csv"
        )
else:
    st.info("📌 Upload PDF files to begin compliance extraction.")

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &


In [ ]:
!pkill -f streamlit
!streamlit run app.py &>/content/logs.txt &


In [ ]:
from pyngrok import ngrok

# Extract only the authtoken string from the command.
# Replace this placeholder with your actual ngrok authtoken.
ngrok.set_auth_token("38ntJpmZNVdjph0TjNk3CHHBGgM_5gzmTF4s8D4wVXmeEeoLP")
public_url = ngrok.connect(8501)
print("✅ Streamlit Dashboard Link:", public_url)

✅ Streamlit Dashboard Link: NgrokTunnel: "https://unwrinkleable-infertilely-tomasa.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
ngrok.kill()


In [ ]:
public_url = ngrok.connect(8501)
print(public_url)


NgrokTunnel: "https://unwrinkleable-infertilely-tomasa.ngrok-free.dev" -> "http://localhost:8501"
